# Sensitivity Analysis: Effect of Lambda Parameter in MESS Angular Distance

This notebook investigates how the lambda parameter in the linear programming problem affects the performance of MESS with angular distance. We test lambda values from 0.0001 to 100 across different M values on a GP regression problem with D=5 dimensions.

In [ ]:
import sys
import os
import time

# Get absolute path to src directory (go up from notebooks to repo root)
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
src_path = os.path.join(repo_root, 'src')
sys.path.insert(0, src_path)

print(f"Repo root: {repo_root}")
print(f"Added to path: {src_path}")

import numpy as np
import matplotlib.pyplot as plt
from mess.data.gp_regression import generate_gp_regression_data
from mess.problems.gp_regression import GaussianProcessRegression
from mess.algorithms.mess import mess_step


## Experiment Setup


In [ ]:
# Problem parameters
D = 5  # Dimension
num_data = 200
length_scale = 1.0
noise_variance = 0.09

# Sampler parameters
n_iters = 500  # Number of MCMC iterations per experiment
burn_in = 100
seed = 42

# Subset of M values to test (don't need all of them)
M_values = [2, 10, 50]

# Lambda values to test (log scale from 0.0001 to 100)
lambda_values = np.logspace(-4, 2, 13)  # 13 values from 10^-4 to 10^2

print(f"Experiment setup:")
print(f"  D = {D}")
print(f"  num_data = {num_data}")
print(f"  n_iters = {n_iters}")
print(f"  M values: {M_values}")
print(f"  Lambda values: {lambda_values}")


## Generate GP Regression Data


In [ ]:
data = generate_gp_regression_data(
    num_data=num_data,
    num_dims=D,
    length_scale=length_scale,
    noise_variance=noise_variance,
    seed=seed,
)

X = data["X"]
y = data["y"]
x0 = data["f_init"]

print(f"Data shapes:")
print(f"  X: {X.shape}")
print(f"  y: {y.shape}")
print(f"  x0: {x0.shape}")

# Create the problem
problem = GaussianProcessRegression(
    X=X,
    y=y,
    length_scale=length_scale,
    noise_variance=noise_variance,
)
print(f"Initial log-likelihood: {problem.log_likelihood(x0):.4f}")


## Lambda Sensitivity Analysis

Run MESS with angular distance for different combinations of M and lambda values.


In [ ]:
# Store results in a structured way
results = {
    'M_values': M_values,
    'lambda_values': lambda_values,
    'mean_intervals': {},  # {M: [mean_int for each lambda]}
    'std_intervals': {},   # {M: [std_int for each lambda]}
    'median_intervals': {},  # {M: [median_int for each lambda]}
    'times': {},           # {M: [time for each lambda]}
}

# Run experiments
for M in M_values:
    print(f"\n{'='*70}")
    print(f"Testing M = {M}")
    print('='*70)
    
    mean_intervals_list = []
    std_intervals_list = []
    median_intervals_list = []
    times_list = []
    
    for lam in lambda_values:
        print(f"  lambda = {lam:.4e}...", end=' ', flush=True)
        
        # Initialize RNG and chain storage
        rng = np.random.default_rng(seed)
        intervals = np.zeros(n_iters + 1, dtype=int)
        intervals[0] = 0  # Initial state at iteration 0
        x = x0.copy()
        
        # Run MCMC with this lambda value
        t0 = time.time()
        for t in range(n_iters):
            x, nr_intervals, _ = mess_step(x, problem, rng, M=M, use_lp=True, 
                                           distance_metric='angular', lam=lam)
            intervals[t + 1] = nr_intervals
        elapsed = time.time() - t0
        
        # Compute statistics (excluding burn-in)
        intervals_post_burnin = intervals[burn_in + 1:]
        mean_int = np.mean(intervals_post_burnin)
        std_int = np.std(intervals_post_burnin)
        median_int = np.median(intervals_post_burnin)
        
        mean_intervals_list.append(mean_int)
        std_intervals_list.append(std_int)
        median_intervals_list.append(median_int)
        times_list.append(elapsed)
        
        print(f"mean={mean_int:.4f}, median={median_int:.1f}, time={elapsed:.2f}s")
    
    # Store results for this M

## Plot 1: Lambda Effect on Mean Shrinking Steps

Show how mean number of intervals varies with lambda for each M value.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

colors = ['blue', 'red', 'green', 'orange']

for idx, M in enumerate(M_values):
    ax = axes[idx]
    
    # Plot mean with error bars
    ax.errorbar(lambda_values, results['mean_intervals'][M], 
                yerr=results['std_intervals'][M], 
                fmt='o-', capsize=5, capthick=2, linewidth=2, markersize=6,
                color=colors[idx], alpha=0.7)
    
    ax.set_xlabel('Lambda (log scale)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean Number of Shrinking Steps', fontsize=11, fontweight='bold')
    ax.set_title(f'M = {M}', fontsize=12, fontweight='bold')
    ax.set_xscale('log')
    ax.grid(True, alpha=0.3, which='both')
    
    # Add vertical line at lambda=0.05 (default value)
    ax.axvline(x=0.05, color='gray', linestyle='--', alpha=0.5, linewidth=1, label='λ=0.05 (default)')
    ax.legend(fontsize=9)

plt.suptitle('Effect of Lambda on Mean Number of Shrinking Steps (with std dev)', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# Find optimal lambda for each M
print("\nOptimal Lambda Values (minimizing mean shrinking steps):")
print("="*60)
for M in M_values:
    best_idx = np.argmin(results['mean_intervals'][M])
    best_lambda = lambda_values[best_idx]
    best_mean = results['mean_intervals'][M][best_idx]
    print(f"  M = {M:4d}: λ = {best_lambda:.4e}, mean steps = {best_mean:.4f}")
print("="*60)


## Plot 2: Combined Lambda Sensitivity Curves


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

colors = ['blue', 'red', 'green', 'orange']
markers = ['o', 's', '^', 'D']

for idx, M in enumerate(M_values):
    ax.plot(lambda_values, results['mean_intervals'][M], 
            marker=markers[idx], linewidth=2.5, markersize=7, 
            color=colors[idx], alpha=0.8, label=f'M = {M}')

# Add vertical line at default lambda value
ax.axvline(x=0.05, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='λ=0.05 (default)')

ax.set_xlabel('Lambda (log scale)', fontsize=13, fontweight='bold')
ax.set_ylabel('Mean Number of Shrinking Steps', fontsize=13, fontweight='bold')
ax.set_title('Lambda Sensitivity: Effect on Mean Shrinking Steps Across Different M Values', 
             fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()


## Plot 3: Heatmap of Results

Visualize how lambda and M jointly affect the mean number of shrinking steps.


In [ ]:
# Create a matrix for heatmap
heatmap_data = np.zeros((len(M_values), len(lambda_values)))

for i, M in enumerate(M_values):
    heatmap_data[i, :] = results['mean_intervals'][M]

fig, ax = plt.subplots(figsize=(14, 6))

im = ax.imshow(heatmap_data, aspect='auto', cmap='viridis', origin='lower')

# Set ticks
ax.set_xticks(np.arange(len(lambda_values)))
ax.set_yticks(np.arange(len(M_values)))

# Set labels
lambda_labels = [f'{lam:.0e}' for lam in lambda_values]
m_labels = [str(M) for M in M_values]

ax.set_xticklabels(lambda_labels, rotation=45, ha='right')
ax.set_yticklabels(m_labels)

ax.set_xlabel('Lambda (log scale)', fontsize=12, fontweight='bold')
ax.set_ylabel('M (Number of Proposals)', fontsize=12, fontweight='bold')
ax.set_title('Heatmap: Mean Number of Shrinking Steps vs Lambda and M', fontsize=13, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Mean Shrinking Steps', rotation=270, labelpad=20, fontweight='bold')

# Add text annotations
for i in range(len(M_values)):
    for j in range(len(lambda_values)):
        text = ax.text(j, i, f'{heatmap_data[i, j]:.2f}',
                      ha="center", va="center", color="white", fontsize=8)

plt.tight_layout()
plt.show()


## Plot 4: Computation Time vs Lambda


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

colors = ['blue', 'red', 'green', 'orange']

for idx, M in enumerate(M_values):
    ax = axes[idx]
    
    ax.plot(lambda_values, results['times'][M], 
            marker='o', linewidth=2.5, markersize=7,
            color=colors[idx], alpha=0.7)
    
    ax.set_xlabel('Lambda (log scale)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Computation Time (seconds)', fontsize=11, fontweight='bold')
    ax.set_title(f'M = {M}', fontsize=12, fontweight='bold')
    ax.set_xscale('log')
    ax.grid(True, alpha=0.3, which='both')
    
    # Add vertical line at lambda=0.05
    ax.axvline(x=0.05, color='gray', linestyle='--', alpha=0.5, linewidth=1, label='λ=0.05 (default)')
    ax.legend(fontsize=9)

plt.suptitle('Computation Time vs Lambda for Different M Values', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()


## Summary Statistics Table


In [ ]:
print("\nDetailed Results Table:")
print("="*100)

for M in M_values:
    print(f"\n{'M = ' + str(M):-^100}")
    print(f"{'Lambda':<15} {'Mean Steps':<15} {'Std Dev':<15} {'Median':<15} {'Time (s)':<15}")
    print("-"*100)
    
    for i, lam in enumerate(lambda_values):
        mean = results['mean_intervals'][M][i]
        std = results['std_intervals'][M][i]
        median = results['median_intervals'][M][i]
        time = results['times'][M][i]
        
        print(f"{lam:<15.4e} {mean:<15.4f} {std:<15.4f} {median:<15.1f} {time:<15.4f}")

print("\n" + "="*100)

# Summary of optimal lambda for each M
print("\nOptimal Lambda (minimizing mean shrinking steps):")
print("="*60)
for M in M_values:
    best_idx = np.argmin(results['mean_intervals'][M])
    best_lambda = lambda_values[best_idx]
    best_mean = results['mean_intervals'][M][best_idx]
    default_mean = results['mean_intervals'][M][np.argmin(np.abs(lambda_values - 0.05))]
    improvement = (default_mean - best_mean) / default_mean * 100
    
    print(f"M = {M:4d}: λ_opt = {best_lambda:.4e}, improvement over λ=0.05: {improvement:+.1f}%")
print("="*60)
